# MLP: group ablation, attention, time of day and latent-space analysis

Use the existing preprocessing/feature bundle. Colab branch **main**, run **run_02**.
This study holds `diversity_ratio` and `habit_score` out of every configuration; it does not repeat their individual feature-selection experiments.

Questions: which input groups matter; what attention adds; whether paired hour sine/cosine inputs help at different hours; which scalar features/embedding groups explain scores; and how clicked versus unclicked candidates relate to user representations.

Decisions use validation mean nDCG@5 across seeds. Test reports the frozen experiments; it never selects features, seeds, hours or models. Hypotheses are not findings. Hourly ranking is not publication freshness: MIND lacks publication timestamps. Cold articles mean no previous clicks, not necessarily newly published articles.


## 1. Shared pipeline and Colab setup


In [ ]:
# Configure this cell before running on Google Colab.
import os
import sys
import subprocess
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    REPO_URL = "https://github.com/nguyenvanminh281005/bt4222-group12.git"
    BRANCH = "main"
    RUN_NAME = "run_02"
    BASE = Path("/content/drive/MyDrive/BT4222 - Group 12")
    CHECKOUT = Path("/content/bt4222_checkout")
    drive.mount("/content/drive")
    if not CHECKOUT.exists():
        subprocess.run([
            "git", "clone", "--branch", BRANCH, "--single-branch",
            REPO_URL, str(CHECKOUT)
        ], check=True)
    active_branch = subprocess.check_output(
        ["git", "-C", str(CHECKOUT), "branch", "--show-current"], text=True
    ).strip()
    if active_branch != BRANCH:
        raise ValueError("Checkout branch differs from BRANCH. Use a fresh Colab runtime.")
    CODE_DIR = (CHECKOUT if (CHECKOUT / "utils/data_utils.py").is_file()
                else CHECKOUT / "bt4222-group12")
    if not (CODE_DIR / "utils/data_utils.py").is_file():
        raise FileNotFoundError("The selected branch does not contain the Group 12 project.")
    os.environ.update({
        "BT4222_BASE": str(BASE),
        "MIND_DATA_DIR": str(BASE / "data"),
        "BT4222_PROCESSED_DIR": str(BASE / "processed"),
        "BT4222_FEATURE_DIR": str(BASE / "features"),
        "BT4222_GLOVE_PATH": str(BASE / "data/glove/glove.6B.300d.txt"),
        "BT4222_OUTPUT_DIR": str(BASE / "runs" / RUN_NAME),
    })
    os.chdir(CODE_DIR)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-r", "requirements.txt"
    ], check=True)
    print("Code:", CODE_DIR)
    print("Features:", os.environ["BT4222_FEATURE_DIR"])
    print("Run outputs:", os.environ["BT4222_OUTPUT_DIR"])


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "utils" / "data_utils.py").is_file()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.data_utils import project_paths, load_preprocessed, load_table, save_table

if os.environ.get("BT4222_USE_DRIVE") == "1":
    from google.colab import drive

    drive.mount("/content/drive")

PATHS = project_paths(PROJECT_ROOT)
from utils.data_utils import source_fingerprint, validate_features, bind_feature_run

feature_id = validate_features(PATHS)
bind_feature_run(PATHS, feature_id)


In [ ]:
from utils.metrics import ndcg_at_k, history_groups


In [ ]:
import os, json, time, copy, random
import numpy as np
import pandas as pd
from scipy.stats import rankdata
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as Fnn
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", device)
FEATURE_DIR = str(PATHS["features"])
OUT_DIR = str(PATHS["outputs"] / "mlp" / "comprehensive_ablation")
os.makedirs(OUT_DIR, exist_ok=True)


## 2. Fixed experiment settings
Three seeds quantify training variation. A full run is 14 configurations × 3 seeds, each up to eight epochs. Completed runs resume only with matching feature, code and training identities. For a smoke run, change seeds/epochs explicitly; do not report smoke metrics as project results.


In [ ]:
# Improved MLP configuration
SEED = 42
EMBED_DIM = 64  # every input block is projected to 64 numbers
# Learned category and subcategory vectors use the original 32/32 configuration.
CAT_DIM = 32
SUBCAT_DIM = 32  # Category and subcategory vectors both use 32 dimensions.
HIDDEN = [128, 64, 32]  # layers of the scoring MLP
DROPOUT = 0.2
ID_DROPOUT = 0.1  # randomly hides an article's subcategory during training
LR = 1e-3
WEIGHT_DECAY = 1e-5
BATCH_SIZE = 32  # impressions per training batch (1 click + 4 skipped articles each)
MAX_EPOCHS = 8
PATIENCE = 2  # stop after 2 epochs without a better validation nDCG@5
COLD_SIM_RATE = 0.15  # share of warm users shown as cold users during training
DROP_FEATURES = [
    "diversity_ratio",
    "habit_score",
]  # fixed exclusions for this study, not a new selection claim
NDCG_KS, RECALL_KS = (5, 10), (5, 10)
# Attention configuration
HIST_LEN = 50  # how many of the user's most recent clicks the attention looks at
ATT_HIDDEN = 32  # size of the small network that scores "how related is this click?"
EVAL_ROWS = 8192  # candidates per evaluation batch (keeps memory low on long slates)
SPLITS = ["train", "val", "test"]
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.width", 200)

STUDY_SEEDS = [42, 43, 44]
RUN_SHAP = True
SHAP_ROWS = 48
SHAP_BACKGROUND = 12
SHAP_NSAMPLES = 256
LATENT_IMPRESSIONS = 600
BOOTSTRAP_REPEATS = 1000
from IPython.display import display
from sklearn.decomposition import PCA
import hashlib
from pathlib import Path
FIG_DIR = Path(OUT_DIR) / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / name, dpi=180, bbox_inches="tight")
    plt.show()


## 3. Load the existing feature exports (no raw data or upstream notebook execution)


In [ ]:
def load_table(name):
    path = os.path.join(FEATURE_DIR, name)
    return (
        pd.read_parquet(path + ".parquet")
        if os.path.exists(path + ".parquet")
        else pd.read_pickle(path + ".pkl")
    )


with open(os.path.join(FEATURE_DIR, "feature_meta.json")) as f:
    meta = json.load(f)

N_ART, N_CAT, N_SUBCAT = (
    meta["dims"]["n_articles"],
    meta["dims"]["n_cat"],
    meta["dims"]["n_subcat"],
)
SPARSE_MAX = meta["cold_start"][
    "sparse_user_max"
]  # users with 1..SPARSE_MAX-1 clicks are "sparse"
# Articles: lookup tables indexed by news_idx (row 0 = padding)
articles = load_table("articles")
content_emb = np.load(os.path.join(FEATURE_DIR, "article_content_emb.npy")).astype(
    np.float32
)
kg_emb = np.load(os.path.join(FEATURE_DIR, "kg_entity_emb.npy")).astype(np.float32)
has_kg = np.zeros(N_ART, np.float32)
has_kg[articles.news_idx.values] = articles.has_kg_entity.values
art_cat = np.zeros(N_ART, np.int64)
art_cat[articles.news_idx.values] = articles.category_idx.values
art_subcat = np.zeros(N_ART, np.int64)
art_subcat[articles.news_idx.values] = articles.subcategory_idx.values
news_ids = np.empty(N_ART, dtype=object)
news_ids[0] = "<PAD>"
news_ids[articles.news_idx.values] = articles.news_id.values
print("articles             :", len(articles))
print("text vectors         :", content_emb.shape)
print("entity vectors       :", kg_emb.shape)


In [ ]:
# Users: one row per distinct click history (profile_idx 0 = no history)
profiles = load_table("user_profiles").sort_values("profile_idx").reset_index(drop=True)
user_emb = np.load(os.path.join(FEATURE_DIR, "user_history_emb.npy")).astype(np.float32)
user_aff = np.load(os.path.join(FEATURE_DIR, "user_category_affinity.npy")).astype(
    np.float32
)
print("user profiles        :", len(profiles))
print("average user vectors :", user_emb.shape)
print("category affinities  :", user_aff.shape)


In [ ]:
# Impressions and candidates for each split
data = {
    s: {
        "imp": load_table(f"{s}/impressions"),
        "groups": load_table(f"{s}/groups"),
        "cands": load_table(f"{s}/candidates"),
    }
    for s in SPLITS
}

for s in SPLITS:
    days = sorted(int(x) for x in data[s]["imp"].day.unique())
    print(
        f"{s:5s}: {len(data[s]['groups']):>8,} impressions | {len(data[s]['cands']):>10,} candidates | days {days}"
    )

print("Split:", meta["split"]["scheme"])
print("Training negatives:", meta["negative_sampling"]["train"])


In [ ]:
prepared_inputs = pd.read_pickle(Path(FEATURE_DIR) / "mlp_inputs.pkl")
A = prepared_inputs["A"]
cand_mu, cand_sd = prepared_inputs["cand_mu"], prepared_inputs["cand_sd"]
user_mu, user_sd = prepared_inputs["user_mu"], prepared_inputs["user_sd"]
ctx_mu, ctx_sd = prepared_inputs["ctx_mu"], prepared_inputs["ctx_sd"]
user_x_all = prepared_inputs["user_x_all"]


In [ ]:
F = meta["features"]
POP_COLS = F["popularity"]  # article popularity on earlier days
INTER_COLS = F["user_x_article"]  # how well the article matches the user
CAND_COLS = POP_COLS + INTER_COLS
USER_COLS = F["user"]["scalar"] + F["optional"]["user"]
CTX_COLS = F["context"] + F["optional"]["context"]  # time of day
BINARY = {"is_cold_article", "category_match", "has_history"}
used = set(CAND_COLS + USER_COLS + CTX_COLS) | {
    "article_content_emb",
    "kg_entity_emb",
    "category_idx",
    "subcategory_idx",
}
assert not used & set(
    meta["excluded_features"]
), "model contract: an excluded feature is in use"
USE_GROUPS = {
    "article": True,
    "popularity": True,
    "user": True,
    "user_x_article": True,
    "context": True,
}


def make_config(name, groups=None, drop=()):
    return {
        "name": name,
        "groups": dict(groups or USE_GROUPS),
        "drop": sorted(set(drop)),
    }


def active_features(cfg):
    keep = lambda cols: [c for c in cols if c not in cfg["drop"]]
    g = cfg["groups"]
    return {
        "popularity": keep(POP_COLS) if g["popularity"] else [],
        "user_x_article": keep(INTER_COLS) if g["user_x_article"] else [],
        "user": keep(USER_COLS) if g["user"] else [],
        "context": keep(CTX_COLS) if g["context"] else [],
    }


CFG = make_config("no_optional", USE_GROUPS, drop=DROP_FEATURES)

for block, cols in active_features(CFG).items():
    print(f"{block:15s}: {cols}")


In [ ]:
assert A["train"]["n"].max() <= 5

for s in SPLITS:
    hl = A[s]["hist_len"]
    print(
        f"{s:5s}: cold users {np.mean(hl == 0):.2%} | sparse users {np.mean((hl > 0) & (hl < SPARSE_MAX)):.2%} | "
        f"candidates that are cold articles {A[s]['is_cold_article'].mean():.2%}"
    )


In [ ]:
COLD_PROFILE = 0
assert (
    profiles.loc[COLD_PROFILE, "history_length"] == 0
), "profile 0 must be the empty-history profile"
COLD_USER_X = user_x_all[COLD_PROFILE].copy()
COLD_INTER = np.array(
    [(meta["imputation_means"][c] - cand_mu[c]) / cand_sd[c] for c in INTER_COLS],
    np.float32,
)
tr = A["train"]
tr_cold = tr["hist_len"] == 0

if tr_cold.any():
    real_cold_rows = np.repeat(tr_cold, tr["n"])
    assert np.allclose(
        tr["cand_x"][real_cold_rows][:, len(POP_COLS) :], COLD_INTER, atol=1e-4
    )
    assert np.allclose(tr["user_x"][tr_cold], COLD_USER_X, atol=1e-5)

print("Check passed: simulated cold-user inputs = real cold-user inputs")
print(
    f"Real cold users in training: {tr_cold.mean():.2%} -> with simulation about "
    f"{tr_cold.mean() + COLD_SIM_RATE * (~tr_cold).mean():.1%} look cold in each epoch"
)


In [ ]:
HIST = np.load(Path(FEATURE_DIR) / "user_history_indices.npy")
assert HIST.shape == (len(profiles), HIST_LEN)
assert (HIST[COLD_PROFILE] == 0).all()


## 4. Metrics, projections and reference MLP
Preserve the existing family evaluator, including mean reciprocal rank across clicked items. Validation/test always use complete observed slates. Ablations retrain the ranker; they do not zero a feature at inference.


In [ ]:
METRIC_NAMES = (
    ["AUC", "MRR"] + [f"nDCG@{k}" for k in NDCG_KS] + [f"Recall@{k}" for k in RECALL_KS]
)


def impression_metrics(labels, scores):
    order = np.argsort(-scores, kind="stable")
    y = labels[order]
    n_pos = y.sum()
    ranks = np.arange(1, len(y) + 1)
    disc = 1.0 / np.log2(ranks + 1)
    out = {"MRR": float((y / ranks).sum() / n_pos)}
    for k in NDCG_KS:
        ideal = disc[: min(int(n_pos), k)].sum()
        out[f"nDCG@{k}"] = ndcg_at_k(labels, y, k)
    for k in RECALL_KS:
        out[f"Recall@{k}"] = float(y[:k].sum() / n_pos)
    n_neg = len(y) - n_pos
    if n_neg > 0:
        r = rankdata(scores)
        out["AUC"] = float(
            (r[labels == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
        )
    else:
        out["AUC"] = np.nan
    return out


def evaluate(Ad, scores):
    """Returns (table of metrics per slice, metrics per impression)."""
    rows, cold_rows = [], []
    for g in range(len(Ad["n"])):
        s, k = Ad["start"][g], Ad["n"][g]
        lab, sc = Ad["label"][s : s + k], scores[s : s + k]
        rows.append(impression_metrics(lab, sc))
        cold = Ad["is_cold_article"][s : s + k] == 1
        cold_pos = (lab == 1) & cold
        if cold_pos.any():
            keep = ~((lab == 1) & ~cold)
            m = impression_metrics(cold_pos[keep].astype(np.float32), sc[keep])
            m["g"] = g
            cold_rows.append(m)
    per_imp = pd.DataFrame(rows)
    hl = Ad["hist_len"]
    slices = {
        "all": np.ones(len(hl), bool),
        "cold_user": hl == 0,
        "sparse_user": (hl > 0) & (hl < SPARSE_MAX),
        "warm_user": hl >= SPARSE_MAX,
    }
    table = []
    for name, m in slices.items():
        sub = per_imp[m]
        table.append(
            {
                "slice": name,
                "n_impressions": int(m.sum()),
                "n_users": int(pd.unique(Ad["user_id"][m]).size),
                **{k: sub[k].mean() for k in METRIC_NAMES},
            }
        )
    cold_df = pd.DataFrame(cold_rows)
    n_cold_art_users = (
        int(pd.unique(Ad["user_id"][cold_df["g"].values]).size) if len(cold_df) else 0
    )
    table.append(
        {
            "slice": "cold_article",
            "n_impressions": len(cold_df),
            "n_users": n_cold_art_users,
            **{
                k: (cold_df[k].mean() if len(cold_df) else np.nan) for k in METRIC_NAMES
            },
        }
    )
    return pd.DataFrame(table), per_imp


# check against a hand-worked example: clicks at ranks 1 and 3 out of 4
m = impression_metrics(
    np.array([1, 0, 1, 0], np.float32), np.array([0.9, 0.8, 0.7, 0.1])
)
assert (
    abs(m["MRR"] - (1 + 1 / 3) / 2) < 1e-9
    and abs(m["AUC"] - 0.75) < 1e-9
    and m["Recall@5"] == 1.0
)
assert abs(m["nDCG@5"] - (1 + 1 / np.log2(4)) / (1 + 1 / np.log2(3))) < 1e-9
print("Metrics match the hand-worked example")


In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class Proj(nn.Module):
    """Linear layer + LayerNorm: turns one input group into a 64-number block."""

    def __init__(self, d_in, d_out):
        super().__init__()
        self.lin, self.ln = nn.Linear(d_in, d_out), nn.LayerNorm(d_out)

    def forward(self, x):
        return self.ln(self.lin(x))


def make_head(d_in):
    """The scoring MLP: d_in -> 128 -> 64 -> 32 -> 1 score."""
    layers = []
    for h in HIDDEN:
        layers += [nn.Linear(d_in, h), nn.ReLU(), nn.Dropout(DROPOUT)]
        d_in = h
    layers.append(nn.Linear(d_in, 1))
    return nn.Sequential(*layers)


In [ ]:
class MLPRanker(nn.Module):
    """Feature-selected MLP ranker."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        g, act = cfg["groups"], active_features(cfg)
        # lookup tables, stored on the GPU with the model
        for name, arr in {
            "content_emb": content_emb,
            "kg_emb": kg_emb,
            "has_kg": has_kg,
            "art_cat": art_cat,
            "art_subcat": art_subcat,
            "user_emb": user_emb,
            "user_aff": user_aff,
        }.items():
            self.register_buffer(name, torch.as_tensor(arr), persistent=False)
        idx = {
            "pop_idx": [CAND_COLS.index(c) for c in act["popularity"]],
            "inter_idx": [CAND_COLS.index(c) for c in act["user_x_article"]],
            "user_idx": [USER_COLS.index(c) for c in act["user"]],
            "ctx_idx": [CTX_COLS.index(c) for c in act["context"]],
        }
        for name, cols in idx.items():
            self.register_buffer(
                name, torch.as_tensor(cols, dtype=torch.long), persistent=False
            )
        self.on = {
            "article": g["article"],
            "popularity": len(idx["pop_idx"]) > 0,
            "user_x_article": len(idx["inter_idx"]) > 0,
            "user": g["user"],
            "user_scalar": len(idx["user_idx"]) > 0,
            "context": len(idx["ctx_idx"]) > 0,
        }
        d = EMBED_DIM
        self.cat_emb = nn.Embedding(N_CAT, CAT_DIM, padding_idx=0)
        n_blocks = 0
        if self.on["article"]:
            self.subcat_emb = nn.Embedding(N_SUBCAT, SUBCAT_DIM, padding_idx=0)
            self.text_proj = Proj(content_emb.shape[1], d)  # MiniLM text 384 -> 64
            self.kg_proj = Proj(kg_emb.shape[1] + 1, d)  # entities 100 + flag -> 64
            self.cat_proj = Proj(
                CAT_DIM + SUBCAT_DIM, d
            )  # category + subcategory -> 64
            n_blocks += 3
        if self.on["popularity"]:
            self.pop_proj = Proj(len(idx["pop_idx"]), d)
            n_blocks += 1
        if self.on["user"]:
            self.user_text_proj = Proj(
                user_emb.shape[1], d
            )  # average clicked-article vector -> 64
            self.user_cat_proj = Proj(CAT_DIM, d)  # category preferences -> 64
            n_blocks += 2
            if self.on["user_scalar"]:
                self.user_num_proj = Proj(len(idx["user_idx"]), d)
                n_blocks += 1
        if self.on["user_x_article"]:
            self.inter_proj = Proj(len(idx["inter_idx"]), d)
            n_blocks += 1
        if self.on["context"]:
            self.ctx_proj = Proj(len(idx["ctx_idx"]), d)
            n_blocks += 1
        self.n_blocks = n_blocks
        self.mlp = make_head(n_blocks * d)

    def _id_dropout(self, idx):
        if self.training and ID_DROPOUT > 0:
            drop = (torch.rand(idx.shape, device=idx.device) < ID_DROPOUT) & (idx > 1)
            idx = idx.masked_fill(drop, 1)
        return idx

    def blocks(self, news, cand_x, prof, user_x, ctx_x):
        """All of 3.3's 64-number input blocks, by name, in 3.3's order."""
        L = news.shape[1]
        per_user = lambda t: t.unsqueeze(1).expand(
            -1, L, -1
        )  # same user block for every candidate
        b = {}
        if self.on["article"]:
            b["text"] = self.text_proj(self.content_emb[news])
            b["kg"] = self.kg_proj(
                torch.cat([self.kg_emb[news], self.has_kg[news].unsqueeze(-1)], -1)
            )
            cat, sub = self._id_dropout(self.art_cat[news]), self._id_dropout(
                self.art_subcat[news]
            )
            b["cat"] = self.cat_proj(
                torch.cat([self.cat_emb(cat), self.subcat_emb(sub)], -1)
            )
        if self.on["popularity"]:
            b["pop"] = self.pop_proj(cand_x.index_select(-1, self.pop_idx))
        if self.on["user_x_article"]:
            b["inter"] = self.inter_proj(cand_x.index_select(-1, self.inter_idx))
        if self.on["user"]:
            b["user_text"] = per_user(self.user_text_proj(self.user_emb[prof]))
            b["user_cat"] = per_user(
                self.user_cat_proj(self.user_aff[prof] @ self.cat_emb.weight)
            )
            if self.on["user_scalar"]:
                b["user_num"] = per_user(
                    self.user_num_proj(user_x.index_select(-1, self.user_idx))
                )
        if self.on["context"]:
            b["ctx"] = per_user(self.ctx_proj(ctx_x.index_select(-1, self.ctx_idx)))
        return b

    def forward(self, news, mask, cand_x, prof, user_x, ctx_x):
        b = self.blocks(news, cand_x, prof, user_x, ctx_x)
        score = self.mlp(torch.cat(list(b.values()), -1)).squeeze(-1)
        return score.masked_fill(~mask, float("-inf"))


In [ ]:
class ClickAttention(nn.Module):
    """Candidate-aware attention over a user's past clicks (DIN-style scoring unit)."""

    def __init__(self, d, hidden):
        super().__init__()
        self.w_q = nn.Linear(d, hidden)  # looks at the candidate
        self.w_k = nn.Linear(d, hidden, bias=False)  # looks at the past click
        self.w_p = nn.Linear(
            d, hidden, bias=False
        )  # looks at candidate x click (shared features)
        self.out = nn.Linear(hidden, 1)  # -> one relevance score

    def forward(self, q, k, valid):
        """
        q:     candidates   [batch, n_candidates, d]
        k:     past clicks  [batch, n_clicks, d]
        valid: real click?  [batch, n_clicks]  (False = padding)
        returns user vector per candidate [batch, n_candidates, d] and weights [batch, n_candidates, n_clicks]
        """
        q4, k4 = q.unsqueeze(2), k.unsqueeze(1)  # line candidates up against clicks
        hidden = torch.relu(self.w_q(q4) + self.w_k(k4) + self.w_p(q4 * k4))
        scores = self.out(hidden).squeeze(-1)  # [batch, n_candidates, n_clicks]
        valid = valid.unsqueeze(1)
        scores = scores.masked_fill(~valid, -1e4)  # padding -> very low score
        weights = (
            torch.softmax(scores, dim=-1) * valid
        )  # add up to 1; no clicks -> all 0
        user_vec = weights @ k  # weighted average of the clicks
        return user_vec, weights


## 5. Unified ranker and precise group-removal semantics
Five groups: article, popularity, user, user×article and context. Context contains exactly `hour_sin` and `hour_cos`, so `no_context` is the paired hour ablation; do not remove only one circular coordinate.

For attention models, `no_article` removes the **direct article blocks** but retains the shared text/category encoder needed by attention; `no_user` removes the **direct user blocks** but retains the attention history. Their effects are conditional on attention being present. The plain-MLP ablations remove those entire direct groups. Category embeddings remain shared with attention/user paths wherever those paths need them.


In [ ]:
class StudyMLP(MLPRanker):
    def __init__(self, cfg, attention=False):
        # Attention must still encode articles when direct article blocks are ablated.
        encoder_cfg = copy.deepcopy(cfg)
        if attention:
            encoder_cfg["groups"]["article"] = True
        super().__init__(encoder_cfg)
        self.study_cfg = copy.deepcopy(cfg)
        self.use_attention = attention
        self.direct_article = cfg["groups"]["article"]
        if attention:
            self.register_buffer("hist", torch.as_tensor(HIST), persistent=False)
            self.attention = ClickAttention(EMBED_DIM, ATT_HIDDEN)
            self.user_attn_proj = Proj(EMBED_DIM, EMBED_DIM)
        count = self.n_blocks - (3 if attention and not self.direct_article else 0)
        self.mlp = make_head((count + int(attention)) * EMBED_DIM)

    def encode_article(self, ids):
        cat = torch.cat([self.cat_emb(self.art_cat[ids]),
                         self.subcat_emb(self.art_subcat[ids])], -1)
        return self.text_proj(self.content_emb[ids]) + self.cat_proj(cat)

    def study_blocks(self, news, cand_x, prof, user_x, ctx_x):
        b = self.blocks(news, cand_x, prof, user_x, ctx_x)
        if self.use_attention:
            clicks = self.hist[prof]
            # Same article projections for candidate and clicked items.
            q = b["text"] + b["cat"]
            u, _ = self.attention(q, self.encode_article(clicks), clicks > 0)
            if not self.direct_article:
                for key in ("text", "kg", "cat"):
                    b.pop(key)
            b["user_attn"] = self.user_attn_proj(u)
        return b

    def forward(self, news, mask, cand_x, prof, user_x, ctx_x):
        b = self.study_blocks(news, cand_x, prof, user_x, ctx_x)
        return self.mlp(torch.cat(list(b.values()), -1)).squeeze(-1).masked_fill(~mask, -float("inf"))

SPECS = {}
for attention in (False, True):
    prefix = "attention" if attention else "mlp"
    for removed in (None, *USE_GROUPS):
        groups = dict(USE_GROUPS)
        if removed:
            groups[removed] = False
        name = prefix + ("_full" if removed is None else "_no_" + removed)
        SPECS[name] = {"cfg": make_config(name, groups, DROP_FEATURES),
                       "attention": attention, "cold_rate": COLD_SIM_RATE}
    name = prefix + "_no_cold_sim"
    SPECS[name] = {"cfg": make_config(name, USE_GROUPS, DROP_FEATURES),
                   "attention": attention, "cold_rate": 0.0}
display(pd.DataFrame([{ "experiment": n, "attention": s["attention"],
    "removed_groups": ", ".join(g for g,v in s["cfg"]["groups"].items() if not v),
    "cold_rate": s["cold_rate"]} for n,s in SPECS.items()]))


## 6. Shared batching, loss and predictions


In [ ]:
def batch_from(Ad, idx, cold_rate=0.0, rng=None):
    idx = np.asarray(idx)
    n, st = Ad["n"][idx], Ad["start"][idx]
    L = int(n.max())
    pos = np.arange(L)
    valid = pos[None, :] < n[:, None]
    rows = np.where(valid, st[:, None] + pos[None, :], 0)
    cand_x = np.where(valid[..., None], Ad["cand_x"][rows], 0.0).astype(np.float32)
    prof, user_x = Ad["prof"][idx].copy(), Ad["user_x"][idx].copy()
    if cold_rate > 0:
        sim = (rng.random(len(idx)) < cold_rate) & (Ad["hist_len"][idx] > 0)
        prof[sim] = COLD_PROFILE
        user_x[sim] = COLD_USER_X
        cand_x[sim, :, len(POP_COLS) :] = COLD_INTER
    to = lambda a, dt=None: torch.as_tensor(a if dt is None else a.astype(dt))
    return {
        "news": to(np.where(valid, Ad["news"][rows], 0)),
        "mask": to(valid),
        "label": to(np.where(valid, Ad["label"][rows], 0.0), np.float32),
        "cand_x": to(cand_x),
        "prof": to(prof),
        "user_x": to(user_x),
        "ctx_x": to(Ad["ctx_x"][idx]),
    }


def train_loader(Ad, seed, cold_rate=COLD_SIM_RATE):
    g = torch.Generator().manual_seed(seed)
    rng = np.random.default_rng(seed + 10_000)
    return DataLoader(
        np.arange(len(Ad["n"])),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=g,
        collate_fn=lambda idx: batch_from(Ad, idx, cold_rate, rng),
    )


def eval_batches(Ad, max_rows=EVAL_ROWS):
    order = np.argsort(
        Ad["n"], kind="stable"
    )  # similar slate lengths together -> little padding
    batches, cur = [], []
    for g in order:
        if cur and (len(cur) + 1) * Ad["n"][g] > max_rows:
            batches.append(np.array(cur))
            cur = []
        cur.append(g)
    batches.append(np.array(cur))
    return batches


In [ ]:
def forward(model, b):
    b = {k: v.to(device, non_blocking=True) for k, v in b.items()}
    return (
        model(b["news"], b["mask"], b["cand_x"], b["prof"], b["user_x"], b["ctx_x"]),
        b,
    )


def listwise_loss(scores, b):
    logp = Fnn.log_softmax(scores, dim=1).masked_fill(~b["mask"], 0.0)
    return (-(logp * b["label"]).sum(1) / b["label"].sum(1).clamp(min=1)).mean()


@torch.no_grad()
def predict(model, Ad):
    """One score per candidate, in the same order as Ad['news']."""
    model.eval()
    out = np.empty(len(Ad["news"]), np.float32)
    for idx in eval_batches(Ad):
        s, _ = forward(model, batch_from(Ad, idx))
        s = s.float().cpu().numpy()
        for j, g in enumerate(idx):
            st, k = Ad["start"][g], Ad["n"][g]
            out[st : st + k] = s[j, :k]
    return out


## 7. Resumable, paired-seed retraining
All configurations use identical training slates, seed list, epoch budget and early-stopping rule. Cold-user simulation has its own paired on/off experiment. Cached weights contain learned parameters only; feature arrays come from the verified bundle. Each checkpoint records its complete spec and a hash of the training implementation.


In [ ]:
RUN_DIR = Path(OUT_DIR) / "runs"
RUN_DIR.mkdir(exist_ok=True)

def make_model(name):
    spec = SPECS[name]
    return StudyMLP(spec["cfg"], spec["attention"]).to(device)

def run_identity(name, seed):
    # inspect source rather than timestamps/absolute paths; changing code invalidates cache.
    import inspect
    implementations = [StudyMLP, MLPRanker, ClickAttention, Proj, make_head,
                       batch_from, train_loader, eval_batches, forward, listwise_loss, predict, fit_run,
                       impression_metrics, evaluate, ndcg_at_k, active_features, set_seed]
    code_id = hashlib.sha256("\n".join(inspect.getsource(x) for x in implementations).encode()).hexdigest()
    return dict(spec=SPECS[name], seed=seed, feature_id=feature_id, code_id=code_id,
        embed_dim=EMBED_DIM, cat_dim=CAT_DIM, subcat_dim=SUBCAT_DIM, hidden=HIDDEN,
        dropout=DROPOUT, id_dropout=ID_DROPOUT, lr=LR, decay=WEIGHT_DECAY,
        batch=BATCH_SIZE, epochs=MAX_EPOCHS, patience=PATIENCE,
        hist_len=HIST_LEN, att_hidden=ATT_HIDDEN)

def fit_run(name, seed):
    set_seed(seed)
    model = make_model(name)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    rows, best, state, bad = [], -np.inf, None, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        total = count = 0
        for b in train_loader(A["train"], seed + epoch, SPECS[name]["cold_rate"]):
            scores, b = forward(model, b)
            loss = listwise_loss(scores, b)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Nonfinite loss: {name}, seed {seed}")
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(b["prof"])
            count += len(b["prof"])
        val = evaluate(A["val"], predict(model, A["val"]))[0].set_index("slice").loc["all"]
        rows.append(dict(epoch=epoch, loss=total/count, **{k:float(val[k]) for k in METRIC_NAMES}))
        print(name, seed, epoch, "validation nDCG@5", round(val["nDCG@5"], 5), flush=True)
        if val["nDCG@5"] > best:
            best, bad = val["nDCG@5"], 0
            state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    if state is None:
        raise RuntimeError("No valid validation checkpoint")
    return state, pd.DataFrame(rows)

def run_or_load(name, seed):
    identity = run_identity(name, seed)
    digest = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:16]
    folder = RUN_DIR / f"{name}_s{seed}_{digest}"
    folder.mkdir(exist_ok=True)
    weights = folder / "weights.pt"
    if not ((folder / "complete.json").is_file() and weights.is_file()):
        state, hist = fit_run(name, seed)
        torch.save(state, weights)
        hist.to_csv(folder / "history.csv", index=False)
        (folder / "complete.json").write_text(json.dumps(identity, indent=2))
    return folder

def restore(name, seed):
    folder = run_or_load(name, seed)
    model = make_model(name)
    model.load_state_dict(torch.load(folder / "weights.pt", map_location=device, weights_only=True))
    model.eval()
    return model

folders = {(name, seed):run_or_load(name, seed) for name in SPECS for seed in STUDY_SEEDS}


## 8. Freeze selection on validation, then report test
Select by mean validation nDCG@5 across all declared seeds. Every completed configuration is considered, including multi-component removals; there is no leave-one-out-only selection shortcut. Diagnostics use the first **declared** seed, not the best seed. Report mean and standard deviation of seed metrics, never treat seed×impression rows as independent observations.


In [ ]:
per_impression, metric_rows = {}, []
for split in ("val", "test"):
    for name in SPECS:
        for seed in STUDY_SEEDS:
            model = restore(name, seed)
            scores = predict(model, A[split])
            table, per = evaluate(A[split], scores)
            per_impression[split, name, seed] = per
            for row in table.to_dict("records"):
                metric_rows.append(dict(split=split, experiment=name, seed=seed, **row))
            del model
    if split == "val":
        val_metrics = pd.DataFrame(metric_rows)
        means = val_metrics.query("slice == 'all'").groupby("experiment")["nDCG@5"].mean()
        SELECTED = means.sort_values(ascending=False, kind="stable").index[0]
        (Path(OUT_DIR) / "selection.json").write_text(json.dumps({"selected":SELECTED,
            "criterion":"mean validation nDCG@5", "seeds":STUDY_SEEDS,
            "validation_means":means.to_dict()}, indent=2))
        print("Frozen validation choice:", SELECTED)

metrics = pd.DataFrame(metric_rows)
metrics.to_csv(Path(OUT_DIR) / "metrics_by_seed.csv", index=False)
summary = metrics.groupby(["split", "experiment", "slice"])[METRIC_NAMES].agg(["mean", "std"])
summary.to_csv(Path(OUT_DIR) / "metrics_summary.csv")
display(summary)


## 9. Group ablation and attention effects, paired within each seed


In [ ]:
paired_rows = []
for split in ("val", "test"):
    for prefix in ("mlp", "attention"):
        for seed in STUDY_SEEDS:
            full = per_impression[split, prefix+"_full", seed]["nDCG@5"].mean()
            for name in SPECS:
                if name.startswith(prefix+"_"):
                    value = per_impression[split, name, seed]["nDCG@5"].mean()
                    paired_rows.append(dict(split=split, experiment=name, seed=seed,
                        baseline=prefix+"_full", delta=value-full))
ablation = pd.DataFrame(paired_rows)
ablation.to_csv(Path(OUT_DIR) / "ablation_paired.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, prefix in zip(axes, ("mlp", "attention")):
    sub = ablation[(ablation.split == "val") & (ablation.baseline == prefix+"_full")]
    agg = sub.groupby("experiment").delta.agg(["mean", "std"]).sort_values("mean")
    ax.barh(agg.index, agg["mean"], xerr=agg["std"].fillna(0))
    ax.axvline(0, color="black", lw=1)
    ax.set(title=prefix+": validation ablation", xlabel="Δ nDCG@5 vs full (mean ± seed SD)")
savefig("group_ablation.png")

attention_rows = []
for split in ("val", "test"):
    for suffix in ("full", "no_context", "no_article", "no_popularity", "no_user", "no_user_x_article", "no_cold_sim"):
        for seed in STUDY_SEEDS:
            for cohort in ("all", "cold_user", "sparse_user", "warm_user"):
                def metric(name):
                    return metrics[(metrics.split == split) & (metrics.experiment == name)
                        & (metrics.seed == seed) & (metrics.slice == cohort)]["nDCG@5"].iloc[0]
                attention_rows.append(dict(split=split, suffix=suffix, seed=seed, cohort=cohort,
                    delta=metric("attention_"+suffix)-metric("mlp_"+suffix)))
attention_effect = pd.DataFrame(attention_rows)
attention_effect.to_csv(Path(OUT_DIR) / "attention_effect.csv", index=False)
display(attention_effect.groupby(["split", "suffix", "cohort"]).delta.agg(["mean", "std"]))


## 10. Time of day: two tables, matched hour ablations and uncertainty
Produce a validation table and a test table, with 24 recorded dataset hours (no timezone conversion). Reconstruct hour from the unstandardised exported sine/cosine features. Include impression counts, cold-user share and cold-article exposure share so changes in cohort composition are visible.

Compare MLP/attention, each with and without both hour coordinates. Bootstrap matched per-impression score differences after averaging over declared seeds. Intervals describe impression-sampling uncertainty, not additional training seeds or evidence of publication freshness. Sparse hours have wider uncertainty; empty hours remain missing.


In [ ]:
HOUR_MODELS = ["mlp_full", "mlp_no_context", "attention_full", "attention_no_context"]

def recorded_hours(split):
    x = A[split]["ctx_x"]
    sin = x[:, CTX_COLS.index("hour_sin")] * ctx_sd["hour_sin"] + ctx_mu["hour_sin"]
    cos = x[:, CTX_COLS.index("hour_cos")] * ctx_sd["hour_cos"] + ctx_mu["hour_cos"]
    hours = np.rint(np.arctan2(sin, cos) * 24 / (2*np.pi)).astype(int) % 24
    assert np.allclose(sin, np.sin(2*np.pi*hours/24), atol=1e-4)
    assert np.allclose(cos, np.cos(2*np.pi*hours/24), atol=1e-4)
    return hours

def paired_ci(values, seed=SEED):
    values = np.asarray(values)
    if len(values) < 2:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    samples = [rng.choice(values, len(values), replace=True).mean() for _ in range(BOOTSTRAP_REPEATS)]
    return tuple(np.quantile(samples, [.025, .975]))

hourly_tables, hourly_deltas = {}, []
for split in ("val", "test"):
    hour = recorded_hours(split)
    Ad = A[split]
    cold_share = np.array([Ad["is_cold_article"][s:s+n].mean() for s,n in zip(Ad["start"], Ad["n"])])
    means = {name:sum(per_impression[split,name,seed][METRIC_NAMES] for seed in STUDY_SEEDS)/len(STUDY_SEEDS)
             for name in HOUR_MODELS}
    rows = []
    for h in range(24):
        mask = hour == h
        row = dict(hour=h, n_impressions=int(mask.sum()),
            cold_user_share=float(np.mean(Ad["hist_len"][mask] == 0)) if mask.any() else np.nan,
            cold_article_share=float(cold_share[mask].mean()) if mask.any() else np.nan)
        for name in HOUR_MODELS:
            for k in ("nDCG@5", "nDCG@10", "MRR", "AUC"):
                row[name+"_"+k] = means[name].loc[mask,k].mean()
        for prefix in ("mlp", "attention"):
            diff = (means[prefix+"_full"]["nDCG@5"] - means[prefix+"_no_context"]["nDCG@5"])[mask]
            lo,hi = paired_ci(diff.to_numpy())
            hourly_deltas.append(dict(split=split, hour=h, architecture=prefix,
                n=int(mask.sum()), delta=diff.mean(), low=lo, high=hi))
        rows.append(row)
    hourly_tables[split] = pd.DataFrame(rows)
    hourly_tables[split].to_csv(Path(OUT_DIR) / f"hourly_{split}.csv", index=False)
    print(split.upper(), "— per-hour model scores")
    display(hourly_tables[split])
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
    for name in HOUR_MODELS:
        axes[0].plot(range(24), hourly_tables[split][name+"_nDCG@5"], marker=".", label=name)
    axes[0].set(ylabel="nDCG@5", title=split+": ranking by recorded hour")
    axes[0].legend(ncol=2)
    axes[1].bar(range(24), hourly_tables[split].n_impressions)
    axes[1].set(xlabel="Recorded dataset hour", ylabel="Impressions", xticks=range(24))
    savefig(f"hourly_{split}.png")

hour_delta = pd.DataFrame(hourly_deltas)
hour_delta.to_csv(Path(OUT_DIR) / "hourly_paired_deltas.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, split in zip(axes, ("val", "test")):
    for prefix in ("mlp", "attention"):
        d = hour_delta[(hour_delta.split == split) & (hour_delta.architecture == prefix)]
        ax.plot(d.hour, d.delta, label=prefix)
        ax.fill_between(d.hour, d.low, d.high, alpha=.15)
    ax.axhline(0, color="black", lw=1)
    ax.set(title=split, xlabel="Recorded hour", ylabel="Δ nDCG@5: with hour − without hour")
    ax.legend()
savefig("hourly_hour_ablation_ci.png")


## 11. SHAP: individual scalar inputs and grouped embedding inputs
Explain the validation-selected model, using the first declared training seed. Explain a uniform sample of validation impressions, one random candidate per impression; use only training impressions for background/reference vectors. Save exact sample indices, labels and values.

Scalar inputs are individual SHAP players. Dense embedding blocks (text, KG, category, user text, user category, attention) are grouped players; each gate replaces an entire block with its training-reference mean. Scalar changes recompute their projection blocks. This explains the ranker's **raw score**, not click probability or nDCG. IDs/history stay fixed for each explanation. Correlated scalars may share attribution and masked combinations may be unrealistic; global mean absolute SHAP is descriptive, not a causal feature-removal result.

KernelExplainer uses a callable and background data: [official SHAP documentation](https://shap.readthedocs.io/en/stable/generated/shap.KernelExplainer.html). Training arrays already standardised by the shared pipeline are used without fitting new statistics. Zero contributions for absent groups must not be interpreted as an importance comparison across different models.


In [ ]:
def candidate_refs(split, size, seed=SEED):
    rng = np.random.default_rng(seed)
    Ad = A[split]
    groups = rng.choice(len(Ad["n"]), min(size, len(Ad["n"])), replace=False)
    slots = np.array([rng.integers(Ad["n"][g]) for g in groups])
    return groups, Ad["start"][groups] + slots

def singleton_batch(split, groups, rows):
    Ad = A[split]
    return {"news":torch.as_tensor(Ad["news"][rows,None], device=device),
        "mask":torch.ones((len(rows),1), dtype=torch.bool, device=device),
        "cand_x":torch.as_tensor(Ad["cand_x"][rows,None,:], device=device),
        "prof":torch.as_tensor(Ad["prof"][groups], device=device),
        "user_x":torch.as_tensor(Ad["user_x"][groups], device=device),
        "ctx_x":torch.as_tensor(Ad["ctx_x"][groups], device=device)}

if RUN_SHAP:
    import shap  # installed by the shared requirements
    model = restore(SELECTED, STUDY_SEEDS[0])
    bg_g, bg_r = candidate_refs("train", max(256, SHAP_BACKGROUND))
    with torch.no_grad():
        bg_b = singleton_batch("train", bg_g, bg_r)
        train_blocks = model.study_blocks(bg_b["news"], bg_b["cand_x"], bg_b["prof"], bg_b["user_x"], bg_b["ctx_x"])
    dense_keys = [k for k in train_blocks if k not in ("pop", "inter", "user_num", "ctx")]
    references = {k:v.mean((0,1)).detach() for k,v in train_blocks.items() if k in dense_keys}
    active = active_features(model.study_cfg)
    scalar_specs = []
    for block, array, columns in (("popularity","cand_x",CAND_COLS),
            ("user_x_article","cand_x",CAND_COLS), ("user","user_x",USER_COLS),
            ("context","ctx_x",CTX_COLS)):
        scalar_specs += [(name,array,columns.index(name)) for name in active[block]]
    names = [x[0] for x in scalar_specs] + ["embedding:"+k for k in dense_keys]

    def scalar_matrix(batch):
        return np.column_stack([batch[arr][...,j].reshape(-1).cpu().numpy() for _,arr,j in scalar_specs])

    background_scalars = scalar_matrix(bg_b)[:SHAP_BACKGROUND]
    background = np.column_stack([background_scalars, np.zeros((len(background_scalars),len(dense_keys)))])
    explain_g, explain_r = candidate_refs("val", SHAP_ROWS, SEED+1)
    attributions, targets, checks = [], [], []
    for g,r in zip(explain_g,explain_r):
        set_seed(SEED + int(g))
        fixed = singleton_batch("val", np.array([g]), np.array([r]))
        target = np.r_[scalar_matrix(fixed)[0], np.ones(len(dense_keys))]

        @torch.no_grad()
        def masked_score(x):
            x = np.asarray(x, np.float32)
            outputs = []
            for start in range(0,len(x),256):
                xx = torch.as_tensor(x[start:start+256], device=device)
                n = len(xx)
                b = {k:v.expand(n,*v.shape[1:]).clone() for k,v in fixed.items()}
                for j,(_,arr,col) in enumerate(scalar_specs):
                    if arr == "cand_x": b[arr][:,0,col] = xx[:,j]
                    else: b[arr][:,col] = xx[:,j]
                blocks = model.study_blocks(b["news"], b["cand_x"], b["prof"], b["user_x"], b["ctx_x"])
                for j,key in enumerate(dense_keys):
                    gate = xx[:,len(scalar_specs)+j,None,None]
                    blocks[key] = references[key][None,None,:] + gate*(blocks[key]-references[key][None,None,:])
                outputs.append(model.mlp(torch.cat(list(blocks.values()),-1)).reshape(-1).cpu().numpy())
            return np.concatenate(outputs)

        explainer = shap.KernelExplainer(masked_score, background, feature_names=names)
        values = np.asarray(explainer.shap_values(target[None,:], nsamples=SHAP_NSAMPLES, l1_reg=0.0)).reshape(-1)
        with torch.no_grad():
            actual = model(**fixed).item()
        assert np.isclose(masked_score(target[None,:])[0],actual,atol=1e-5)
        checks.append(dict(group=int(g), row=int(r), label=float(A["val"]["label"][r]),
            score=actual, expected=float(explainer.expected_value), residual=actual-float(explainer.expected_value)-values.sum()))
        attributions.append(values)
        targets.append(target)
    sv, sx = np.vstack(attributions), np.vstack(targets)
    np.savez(Path(OUT_DIR)/"shap_values.npz", values=sv, inputs=sx, names=np.array(names))
    pd.DataFrame(checks).to_csv(Path(OUT_DIR)/"shap_sample_and_additivity.csv",index=False)
    importance = pd.DataFrame({"feature":names,"mean_abs_shap":np.abs(sv).mean(0)}).sort_values("mean_abs_shap",ascending=False)
    importance.to_csv(Path(OUT_DIR)/"shap_importance.csv",index=False)
    display(importance)
    shap.summary_plot(sv, sx, feature_names=names, plot_type="bar", show=False)
    savefig("shap_global_bar.png")
    scalar_count = len(scalar_specs)
    if scalar_count:
        shap.summary_plot(sv[:,:scalar_count], sx[:,:scalar_count], feature_names=names[:scalar_count], show=False)
        savefig("shap_scalar_beeswarm.png")
    del model


## 12. Latent-space ablation: four matched models and two shared spaces
Compare MLP/attention × with/without hour, using the same sampled warm test impressions. Select one clicked and one exposed unclicked article per impression, without using model scores for sampling. Save exact rows and full-dimensional cosine/distance measures before projection.

**Shared article space (64-d):** encode candidate and each history article with the same text+category encoder. The user is either the mean of those encoded history articles or the attention-weighted mean for that candidate. For plain MLP this mean is a diagnostic construction; it is **not** the model's separately projected `user_text` block. With attention, the weighted vector is the actual pre-projection attention representation.

**Shared category space (32-d):** user category proportions × the model's shared category table, versus the candidate category embedding. This is an actual shared lookup space in both architectures. Do not compare `user_text_proj` and `text_proj` by cosine: their axes are learned separately.

Fit one PCA on pooled vectors within each model+space, use identical coordinates for user/candidate pairs in that panel, and report explained variance. Separate models have independently learned axes; cross-model comparisons use cosine/distance distributions, not absolute PCA coordinates. Two-dimensional clustering alone is not evidence of recommendation quality.


In [ ]:
rng = np.random.default_rng(SEED)
Ad = A["test"]
eligible = []
for g in np.flatnonzero(Ad["hist_len"] >= SPARSE_MAX):
    st,n = Ad["start"][g],Ad["n"][g]
    labels = Ad["label"][st:st+n]
    if (labels == 1).any() and (labels == 0).any(): eligible.append(g)
sample_g = rng.choice(eligible,min(LATENT_IMPRESSIONS,len(eligible)),replace=False)
pair_groups, pair_rows = [], []
for g in sample_g:
    st,n = Ad["start"][g],Ad["n"][g]
    labels = Ad["label"][st:st+n]
    for label in (1,0):
        pair_groups.append(g)
        pair_rows.append(st+rng.choice(np.flatnonzero(labels == label)))
pair_groups,pair_rows = np.array(pair_groups),np.array(pair_rows)
if not len(pair_rows):
    raise ValueError("Latent diagnostics need warm impressions containing clicked and unclicked candidates")
latent_metadata = pd.DataFrame(dict(group=pair_groups,row=pair_rows,label=Ad["label"][pair_rows]))
latent_metadata.to_csv(Path(OUT_DIR)/"latent_sample.csv",index=False)

def unit(x):
    return x / np.maximum(np.linalg.norm(x,axis=-1,keepdims=True),1e-12)

latent_records = []
for name in HOUR_MODELS:
    model = restore(name,STUDY_SEEDS[0])
    vectors = {key:[] for key in ("article_candidate","article_user","category_candidate","category_user")}
    with torch.no_grad():
        for s in range(0,len(pair_rows),128):
            gs,rs = pair_groups[s:s+128],pair_rows[s:s+128]
            b = singleton_batch("test",gs,rs)
            ids = torch.as_tensor(HIST[Ad["prof"][gs]],device=device)
            q,k = model.encode_article(b["news"]),model.encode_article(ids)
            valid = ids > 0
            if model.use_attention:
                u,_ = model.attention(q,k,valid)
            else:
                u = ((k*valid[...,None]).sum(1)/valid.sum(1,keepdim=True).clamp(min=1))[:,None,:]
            vectors["article_candidate"].append(q[:,0].cpu().numpy())
            vectors["article_user"].append(u[:,0].cpu().numpy())
            vectors["category_candidate"].append(model.cat_emb(model.art_cat[b["news"]])[:,0].cpu().numpy())
            vectors["category_user"].append((model.user_aff[b["prof"]] @ model.cat_emb.weight).cpu().numpy())
    vectors = {key:np.concatenate(value) for key,value in vectors.items()}
    np.savez(Path(OUT_DIR)/f"latent_{name}.npz",**vectors)
    fig,axes = plt.subplots(2,2,figsize=(12,9))
    labels = Ad["label"][pair_rows]
    for j,space in enumerate(("article","category")):
        u,c = vectors[space+"_user"],vectors[space+"_candidate"]
        un,cn = unit(u),unit(c)
        cosine = (un*cn).sum(1)
        distance = np.linalg.norm(un-cn,axis=1)
        for label in (1,0):
            mask = labels == label
            latent_records.append(dict(experiment=name, space=space,label=label,n=int(mask.sum()),
                mean_cosine=float(cosine[mask].mean()),std_cosine=float(cosine[mask].std()),
                mean_unit_distance=float(distance[mask].mean()),
                user_dispersion=float(np.mean(np.sum((un[mask]-un[mask].mean(0))**2,axis=1))),
                candidate_dispersion=float(np.mean(np.sum((cn[mask]-cn[mask].mean(0))**2,axis=1)))))
            axes[j,1].hist(cosine[mask],bins=30,alpha=.5,density=True,label="clicked" if label else "unclicked")
        axes[j,1].set(title=space+": full-dimensional similarity",xlabel="Cosine",ylabel="Density")
        axes[j,1].legend()
        # Normalize for directional comparison, then jointly fit PCA.
        pca = PCA(n_components=2)
        xy = pca.fit_transform(np.vstack([un,cn]))
        ux,cx = xy[:len(u)],xy[len(u):]
        axes[j,0].scatter(ux[:,0],ux[:,1],s=5,alpha=.2,label="user")
        for label,color in ((1,"tab:green"),(0,"tab:red")):
            mask = labels == label
            axes[j,0].scatter(cx[mask,0],cx[mask,1],s=6,alpha=.3,c=color,label="clicked" if label else "unclicked")
        for i in range(min(24,len(u))):
            axes[j,0].plot([ux[i,0],cx[i,0]],[ux[i,1],cx[i,1]],alpha=.15,color="gray")
        axes[j,0].set(title=f"{space}: PCA retains {pca.explained_variance_ratio_.sum():.1%}",xlabel="PC1",ylabel="PC2")
        axes[j,0].legend()
    fig.suptitle(name+" — matched candidate/history sample")
    savefig(f"latent_{name}.png")
    del model
latent_summary = pd.DataFrame(latent_records)
latent_summary.to_csv(Path(OUT_DIR)/"latent_summary.csv",index=False)
display(latent_summary)
fig,axes = plt.subplots(1,2,figsize=(13,4))
for ax,space in zip(axes,("article","category")):
    table = latent_summary[latent_summary.space == space].pivot(index="experiment",columns="label",values="mean_cosine")
    table.rename(columns={0:"unclicked",1:"clicked"}).plot.bar(ax=ax)
    ax.set(title=space+": shared-space cosine by ablation",ylabel="Mean cosine")
savefig("latent_ablation_comparison.png")


## 13. Report checklist and output inventory

1. **Groups:** report validation ablation deltas with seed SD; group removal measures information plus changed model capacity, not an isolated causal effect.
2. **Features:** report scalar SHAP ranking separately from grouped embedding attribution; SHAP explains scores, ablation measures retrained ranking performance.
3. **Attention:** report matched comparisons across seeds and user cohorts. It may help warm users and harm cold users; do not assume the sign.
4. **Time:** report both 24-hour tables, paired deltas and sample counts. Improvement by recorded hour supports time-context usefulness, not the ability to detect publication freshness or reduced exposure fatigue.
5. **Latent:** report clicked–unclicked separation and dispersion in the original dimensions alongside PCA. Mean-history and attention-history representations have different definitions; closer vectors alone do not establish better ranking. PCA orientation is not comparable across independently trained models.

All output is under `runs/run_02/mlp/comprehensive_ablation` in Colab. No metrics are prefilled. Re-run this notebook on the prepared run_02 bundle to obtain findings.


In [ ]:
manifest = dict(selected=SELECTED,seeds=STUDY_SEEDS,feature_id=feature_id,
    dropped_features=DROP_FEATURES,experiments=SPECS,
    shap=dict(enabled=RUN_SHAP,split="val",rows=SHAP_ROWS,background=SHAP_BACKGROUND,nsamples=SHAP_NSAMPLES),
    latent=dict(split="test",impressions=int(len(sample_g)),seed=STUDY_SEEDS[0]),
    hourly="recorded dataset hour; both hour sine/cosine removed together")
(Path(OUT_DIR)/"study_manifest.json").write_text(json.dumps(manifest,indent=2))
print("Results:", OUT_DIR)
display(pd.DataFrame({"artifact":sorted(p.name for p in Path(OUT_DIR).iterdir())}))
